# Create Paul Hamlyn Foundation Grant Awards (ORG-LEVEL GRANT PATTERN, Method 1 360Giving open data)

Paul Hamlyn Foundation funds UK civil society, arts, education, youth, migration, and social-justice organizations. This ingest covers the foundation's official 360Giving workbook currently listed in the 360Giving Data Registry: grants awarded between April 2006 and January 2026.

**Method 1 (direct open-data download).** The 360Giving Data Registry lists publisher `Paul Hamlyn Foundation` and resolves to a direct Excel workbook download on `phf.org.uk`. No browser automation or search API is required.

**Awarding body:** Paul Hamlyn Foundation - F4320320621 (GB, ROR 03wztvy18, DOI 10.13039/501100001263)

**Source:** one `grants` sheet. This is an org-level grant funder: each grant is made to a recipient organization, with no named principal investigator in the source. The source provides stable `Identifier` values, title, description, positive GBP amount, real award date, planned grant end date, recipient organization name/identifier, beneficiary country code, and grant programme.

**Schema choices:**
- One row per grant. `funder_award_id` = source `Identifier`, e.g. `360G-phf-19127`.
- `display_name` = source `Title`; `description` = source `Description`.
- `funding_type` = `grant` uniformly; `funder_scheme` = source `Grant Programme:Title`.
- `amount` = positive `Amount Awarded`; `currency` = source `Currency`. Amount coverage is 100%, so runbook section 6.7 is not waived.
- `start_date` = planned start date where present, otherwise real `Award date`; `end_date` = planned end date. These are source-published day-level dates.
- `lead_investigator` = org-only lead: given/family/orcid NULL and `affiliation.name` = recipient organization. `affiliation.country` is NULL because the workbook publishes beneficiary country, not recipient-organization country.
- `landing_page_url` is NULL because the workbook does not expose per-grant pages.

**Contractor handoff:** this notebook is prepared for admin execution after the parquet is uploaded to S3. Contractor has no S3/Databricks access.


## Step 1: Create staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.paul_hamlyn_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/paul_hamlyn/paul_hamlyn_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.paul_hamlyn_raw;


## Step 1.5: Inspect raw coverage, dates, programmes, and amounts

Expected after local validation: 4,198 grants, 2006-2026 start years, 2006-2041 end years, 100% title/description/amount/currency/date/recipient coverage, 99.6% programme coverage, and 99.8% beneficiary-country-code coverage.


In [ ]:
%sql
DESCRIBE openalex.awards.paul_hamlyn_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.paul_hamlyn_raw
LIMIT 5;


In [ ]:
%sql
-- Uniqueness and high-level coverage from raw staging.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    COUNT(title) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(award_date) AS has_award_date,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(start_year) AS has_start_year,
    COUNT(end_year) AS has_end_year,
    COUNT(recipient_org) AS has_recipient_org,
    COUNT(recipient_org_identifier) AS has_recipient_org_identifier,
    COUNT(beneficiary_country_code) AS has_beneficiary_country_code,
    COUNT(grant_programme) AS has_grant_programme,
    COUNT(from_open_call) AS has_open_call,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.paul_hamlyn_raw;


In [ ]:
%sql
-- Programme/year distribution and amount coverage.
SELECT
    grant_programme,
    start_year,
    COUNT(*) AS rows,
    COUNT(amount) AS has_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.paul_hamlyn_raw
GROUP BY grant_programme, start_year
ORDER BY start_year DESC, rows DESC;


In [ ]:
%sql
-- Beneficiary country is preserved for audit, but not used as lead affiliation country.
SELECT beneficiary_country_code, COUNT(*) AS rows, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.paul_hamlyn_raw
GROUP BY beneficiary_country_code
ORDER BY rows DESC;


In [ ]:
%sql
-- Top recipient organizations, to confirm org-level lead parsing.
SELECT recipient_org, COUNT(*) AS rows,
       ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.paul_hamlyn_raw
WHERE recipient_org IS NOT NULL
GROUP BY recipient_org
ORDER BY rows DESC, total_gbp DESC
LIMIT 20;


## Step 1.6: Fail-fast - verify Paul Hamlyn Foundation funder row exists

Runbook section 2.2.4 mandatory check. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320621;


## Step 2: Transform to OpenAlex awards schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.paul_hamlyn_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320621  -- Paul Hamlyn Foundation
), awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        COALESCE(n.title, CONCAT('Paul Hamlyn Foundation grant ', n.funder_award_id)) AS display_name,
        n.description,
        f.funder_id,
        n.funder_award_id,
        CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
        CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN n.currency ELSE NULL END AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'grant' AS funding_type,
        n.grant_programme AS funder_scheme,
        'paul_hamlyn_360giving' AS provenance,
        TRY_CAST(n.start_date AS DATE) AS start_date,
        TRY_CAST(n.end_date AS DATE) AS end_date,
        TRY_CAST(n.start_year AS INT) AS start_year,
        TRY_CAST(n.end_year AS INT) AS end_year,
        CASE
            WHEN n.recipient_org IS NOT NULL THEN struct(
                CAST(NULL AS STRING) AS given_name,
                CAST(NULL AS STRING) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    n.recipient_org AS name,
                    CAST(NULL AS STRING) AS country,
                    CASE
                        WHEN n.recipient_org_identifier IS NOT NULL THEN array(struct(
                            n.recipient_org_identifier AS id,
                            CAST('360Giving Recipient Org:Identifier' AS STRING) AS type,
                            CAST('source' AS STRING) AS asserted_by
                        ))
                        ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                    END AS ids
                ) AS affiliation
            )
            ELSE NULL
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        CAST(NULL AS STRING) AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               TRY_CAST(abs(xxhash64(CONCAT(
                   TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
               ))) % 9000000000 AS STRING)) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM openalex.awards.paul_hamlyn_raw n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
)
SELECT
    * EXCEPT(start_year, end_year),
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE start_year END AS start_year,
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE end_year END AS end_year
FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw at priority 192


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'paul_hamlyn_360giving' AND priority = 192;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    192 AS priority  -- Paul Hamlyn Foundation grants
FROM openalex.awards.paul_hamlyn_awards;


## Step 6: Verification

Amount coverage should be 100% and currency should be GBP. Lead investigator affiliation country should be NULL by design because the source publishes beneficiary country, not recipient-organization country.


In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM openalex.awards.paul_hamlyn_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.paul_hamlyn_awards;


In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(start_year) AS has_start_year,
    COUNT(end_year) AS has_end_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_lead
FROM openalex.awards.paul_hamlyn_awards;


In [ ]:
%sql
-- Runbook section 6.7 amount/currency check. Expected: 4,198/4,198 rows, GBP only.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.paul_hamlyn_awards;


In [ ]:
%sql
-- Duplicate guard.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.paul_hamlyn_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;


In [ ]:
%sql
-- Year sanity. max_start_year should be <= YEAR(current_date()) + 1 after inline cap; future end years are legitimate multi-year grants.
SELECT
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    MIN(end_year) AS min_end_year,
    MAX(end_year) AS max_end_year,
    SUM(CASE WHEN start_year > YEAR(current_date()) + 1 THEN 1 ELSE 0 END) AS future_start_rows
FROM openalex.awards.paul_hamlyn_awards;


In [ ]:
%sql
-- Programme split.
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.paul_hamlyn_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;


In [ ]:
%sql
-- Country sanity: NULL by design; beneficiary country is preserved only in raw staging.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.paul_hamlyn_awards
GROUP BY lead_investigator.affiliation.country
ORDER BY rows DESC;


In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.paul_hamlyn_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
-- Sample rows for admin QA.
SELECT
    id,
    SUBSTRING(display_name, 1, 80) AS title,
    funder_scheme,
    funding_type,
    start_date,
    end_date,
    amount,
    currency,
    lead_investigator.affiliation.name AS recipient_org,
    lead_investigator.affiliation.country AS country
FROM openalex.awards.paul_hamlyn_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;


## Handoff notes

Contractor-complete status: script and notebook are ready, `CreateAwards.ipynb` has priority 192, and the tracker is marked Step 4 in the PR. Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect the verification cells, and only then decide whether to wire job YAML.
